In [1]:
import os 
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
import gradio as gr

In [27]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENROUTER_API_KEY')
claude_api_key = os.getenv('OPENROUTER_API_KEY')
gemini_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exist and begins {openai_api_key[:3]}")
else:
    print(f"OpenAI API Key not set")
if claude_api_key:
    print(f"claude API Key exist and begins {claude_api_key[:3]}")
else:
    print(f"claude API Key not set")
if gemini_api_key:
    print(f"gemini API Key exist and begins {gemini_api_key[:3]}")
else:
    print(f"gemini API Key not set")

OpenAI API Key exist and begins sk-
claude API Key exist and begins sk-
gemini API Key exist and begins sk-


In [57]:
openai_url = "https://openrouter.ai/api/v1"
claude_url = "https://openrouter.ai/api/v1"
gemini_url = "https://openrouter.ai/api/v1"

openai_model = "openai/gpt-4o-mini"
claude_model = "anthropic/claude-sonnet-4"
gemini_model = "google/gemini-2.5-flash"

openai = OpenAI(base_url=openai_url, api_key=openai_api_key)
claude = OpenAI(base_url=claude_url, api_key=claude_api_key)
gemini = OpenAI(base_url=gemini_url, api_key=gemini_api_key)

In [39]:
system_message = "You are a helpful assistant"

def message_gpt(prompt):
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ]
        response = openai.chat.completions.create(model=openai_model, messages=messages)
        return response.choices[0].message.content

In [40]:
message_gpt("What is today date ?")

"Today's date is October 16, 2023."

In [41]:
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [42]:
shout("hello")

Shout has been called with input hello


'HELLO'

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input hello
Shout has been called with input who are you >?


In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input eat
Shout has been called with input i wanna eat burger


## Adding authentication

Gradio makes it very easy to have userids and passwords

Obviously if you use this, have it look properly in a secure place for passwords! At a minimum, use your .env

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("admin", "lutfi"))

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input samsul
Shout has been called with input kamu mau apa ?


## Forcing dark mode

Gradio appears in light mode or dark mode depending on the settings of the browser and computer. There is a way to force gradio to appear in dark mode, but Gradio recommends against this as it should be a user preference (particularly for accessibility reasons). But if you wish to force dark mode for your screens, below is how to do it.

In [50]:
force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme')!== 'light') {
        url.searchParams.set('__theme', 'light');
        window.location.href = url.href
    }
}
"""
gr.Interface(fn=shout, outputs="textbox", inputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input like this color


In [52]:
# Adding a little more:

message_input = gr.Textbox(label="Your Message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=shout,
    title="CHAT BOT",
    inputs=[message_input],
    outputs=[message_output],
    examples=["hello", "world"],
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input i wanna say thank you with you, cz you're so many help me 

Shout has been called with input hello
Shout has been called with input world


In [60]:
# Stream(konsep) = data yang datang secara bertahap, bukan sekaligus
# stream=True = setelan untuk meminta LLM mengirim jawaban/token output secara bertahap
# yield = return
# return = function mengembalikan satu nilai, lalu selesai, function berhenti
# yield = function mengembalikan satu nilai, tapi belum selesai, mengingat posisi dan bisa di lanjutkan untuk mengeluarkan nilai berikutnya
# kenapa pakai yield ? karena streaming

def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    stream = openai.chat.completions.create(
        model=openai_model,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [62]:
message_input = gr.Textbox(label="Your Message", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ],
    flagging_mode="never"    
)

view.launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


In [63]:
def stream_claude(prompt):
    messages =[
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    stream = claude.chat.completions.create(
        model=claude_model,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [65]:
message_input = gr.Textbox(label="Your message", info="Enter a messages for Claude 4.5 Sonnet", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_claude,
    title="CLAUDE",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


# GABUNG KEDUA MODEL DALAM 1 APP

In [74]:
def stream_model(prompt, model):
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="CLAUDE":
        result = stream_claude(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [83]:
message_input = gr.Textbox(label="Your Message", info="Enter a messages for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "CLAUDE"], label="Selec Model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs",
    inputs=[message_input, model_selector],
    outputs=[message_output],
    examples=[
            ["Explain the Transformer architecture to a layperson", "GPT"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "CLAUDE"]
        ],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


# Building a company brochure generator

Now you know how - it's simple!

In [89]:
from scraper import fetch_website_contents

Traceback (most recent call last):
  File "/Users/boss88/llm-project/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 849, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/boss88/llm-project/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/boss88/llm-project/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/boss88/llm-project/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1635, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/boss88/llm-project/llm_engi

In [84]:
# Again this is typical Experimental mindset - I'm changing the global variable we used above:

system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [91]:
def stream_brochure(company, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model == "GPT":
        result = stream_gpt(prompt)
    elif model == "CLAUDE":
        result = stream_claude(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [92]:
name_input = gr.Textbox(label="Company Name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT", "CLAUDE"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator",
    inputs=[name_input, url_input, model_selector],
    outputs=[message_output],
    examples=[
        ["Hugging Face", "https://huggingface.co", "GPT"],
        ["Edward Donner", "https://edwarddonner.com", "CLAUDE"]
    ],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.
